In [1]:
import sys
import tensorflow as tf

print(f"TensorFlow ver is {tf.__version__}")
print(f"PYTHON ver is {sys.version}")
print("GPU is","available" if tf.config.list_physical_devices('GPU') else "NOT AVAILABLE")

TensorFlow ver is 2.10.0
PYTHON ver is 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
GPU is available


# CELL1) IMPORT + PATHS + 기본설정

In [2]:
# %%
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

print("TF:", tf.__version__)
print("GPU:", "available" if tf.config.list_physical_devices('GPU') else "NOT AVAILABLE")

# ------------------------------
# Paths (필요시 여기만 수정)
# ------------------------------
DATA_DIR  = "database"
EVAL_FILE = "cmall_remove_duplicate_m_all.csv"   # 또는 "small_cases.csv"

QUALITY_RES_PATH = "best_model_residue.h5"
QUALITY_BLA_PATH = "best_model_blaine.h5"

POLICY_RES_PATH  = "policy_residue.keras"
POLICY_BLA_PATH  = "policy_blaine.keras"

SCALER_X_RES_PATH = "scaler_x_res_38.pkl"
SCALER_Y_RES_PATH = "scaler_y_res_1.pkl"

SCALER_X_BLA_PATH = "scaler_x_bla_38.pkl"
SCALER_Y_BLA_PATH = "scaler_y_bla_1.pkl"

# ------------------------------
# Evaluation config
# ------------------------------
RANDOM_STATE = 42
BATCH_SIZE   = 1024

DELTA_MAX_DEFAULT = 0.2  # normalized space bound for each control (학습 때와 동일)
N_SKIP_BATCHES = 3
N_BATCHES      = 10

# spec (현장 기준)
RES_MIN, RES_MAX = 7.0, 9.0
BLA_MIN, BLA_MAX = 3700.0, 3900.0

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


TF: 2.10.0
GPU: available


# CELL2) 컬럼 정의 (TRAIN과 동일해야 함)

In [3]:
# %%
CONTROL_COLS_9 = [
    "RP_roller_p1", "RP_roller_p2", "RP_sep_rpm", "RP_sep_fan_damper", "RP_sep_BF_damper",
    "mill_BF_damper", "mill_sep_rpm", "mill_sep_fan_damper", "grind_aid",
]
assert len(CONTROL_COLS_9) == 9

MONITOR_COLS_28 = [
    "RP_spac", "RP_skew", "RP_roller_energy1", "RP_roller_energy2", "RP_roller_vib",
    "RP_BE_energy1", "RP_BE_energy2", "RP_sep_BF_pressure", "dosing_BE_energy",
    "mill_feed_c", "mill_feed_cir", "mill_energy", "mill_in_temp", "mill_out_temp",
    "mill_BE_energy", "mill_out_gas_temp", "mill_out_mater_temp", "mill_BF_pressure",
    "mill_sep_BF_pressure", "mill_sep_BF_fan_damper", "final_BE_1", "final_BE_2",
    "final_mater_temp",
    "feed_clinker", "feed_gypsum", "feed_slag", "feed_FA", "feed_total",
]
assert len(MONITOR_COLS_28) == 28

# ---- Quality input은 [u(9), mon(28), qprev(1)] = 38
# 주의: qprev 컬럼명은 학습 때 사용했던 “이전 품질” 컬럼명과 정확히 같아야 합니다.
# 지금 코드 흐름에서는 df에 residue_prev / blaine_prev가 있다고 가정합니다.
RES_PREV_COL = "residue"
BLA_PREV_COL = "blaine"

# ---- Policy input은 39: [mon(28), qcur(1), u_cur(9), yT(1)]
# Xp_full은 평가 편의상 qprev까지 포함해서 40: +[qprev(1)]
POLICY_INPUT_DIM = 39


# CELL3) 데이터 로드 + 최소 컬럼 체크

In [4]:
# %%
csv_path = os.path.join(DATA_DIR, EVAL_FILE)
df = pd.read_csv(csv_path)

# 현재 품질(qcur) 및 목표(yT) 컬럼
RES_CUR_COL = "residue"
BLA_CUR_COL = "blaine"
RES_TGT_COL = "residue_target"
BLA_TGT_COL = "blaine_target"

required = set(CONTROL_COLS_9 + MONITOR_COLS_28 +
               [RES_PREV_COL, BLA_PREV_COL, RES_CUR_COL, BLA_CUR_COL, RES_TGT_COL, BLA_TGT_COL])

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}")

df = df.dropna(subset=list(required)).reset_index(drop=True)
print("Loaded:", df.shape, "from", csv_path)

display(df.head(19))


Loaded: (60572, 43) from database\cmall_remove_duplicate_m_all.csv


,RP_roller_p1,RP_roller_p2,RP_sep_rpm,RP_sep_fan_damper,RP_sep_BF_damper,mill_BF_damper,mill_sep_rpm,mill_sep_fan_damper,grind_aid,RP_spac,RP_skew,RP_roller_energy1,RP_roller_energy2,RP_roller_vib,RP_BE_energy1,RP_BE_energy2,RP_sep_BF_pressure,dosing_BE_energy,mill_feed_c,mill_feed_cir,mill_energy,mill_in_temp,mill_out_temp,mill_BE_energy,mill_out_gas_temp,mill_out_mater_temp,mill_BF_pressure,mill_sep_BF_pressure,mill_sep_BF_fan_damper,final_BE_1,final_BE_2,final_mater_temp,feed_total,feed_clinker,feed_gypsum,feed_slag,feed_FA,blaine,residue,blaine_prev,residue_prev,blaine_target,residue_target
0,81,89.0,400,100,60,15,1148,100,169.2,44.0,1.0,146.0,135.0,1.3,0,117.0,-65,20,309.0,484.0,173,26.0,31,75,87,93,-117,-239.0,30,89,78,72,210.2,180.5,10.8,10.5,8.4,3590,8.3,3610,8.5,3800,8
1,79,83.0,418,100,60,15,1148,100,169.2,46.0,1.0,148.0,143.0,1.4,0,127.0,-64,22,381.0,602.0,168,29.0,36,77,94,100,-116,-235.0,30,90,79,79,211.1,181.6,10.7,10.5,8.3,3590,8.3,3610,8.5,3800,8
2,74,80.0,417,100,60,15,1149,100,169.2,46.0,0.0,145.0,132.0,1.8,0,124.0,-66,20,336.0,547.0,173,31.0,40,72,96,102,-117,-230.0,30,88,79,81,212.6,182.7,10.8,10.6,8.5,3590,8.3,3610,8.5,3800,8
3,74,78.0,417,100,60,15,1148,100,169.2,46.0,1.0,147.0,142.0,1.7,0,122.0,-65,21,360.0,576.0,163,32.0,42,75,97,103,-118,-236.0,30,88,77,81,213.1,183.1,10.8,10.6,8.6,3590,8.3,3610,8.5,3800,8
4,75,78.0,427,100,60,15,1147,100,169.2,46.0,0.0,148.0,140.0,1.3,0,122.0,-70,19,367.0,576.0,164,35.0,44,76,97,104,-118,-232.0,30,88,77,82,213.7,183.7,10.8,10.6,8.6,3630,8.5,3590,8.3,3800,8
5,75,77.0,427,100,60,15,1148,100,169.2,47.0,1.0,148.0,132.0,1.3,0,125.0,-64,20,337.0,547.0,169,36.0,46,75,97,103,-116,-229.0,30,87,78,82,215.0,184.9,10.9,10.7,8.5,3630,8.5,3590,8.3,3800,8
6,77,82.0,427,100,60,15,1149,100,169.2,46.0,1.0,154.0,146.0,1.2,0,125.0,-64,25,338.0,558.0,162,36.0,46,70,98,104,-116,-226.0,30,90,77,82,216.3,185.9,11.0,10.8,8.6,3630,8.5,3590,8.3,3800,8
7,75,89.0,409,100,60,15,1146,100,169.2,46.0,1.0,149.0,144.0,1.5,0,120.0,-65,23,372.0,611.0,172,38.0,48,71,97,103,-116,-230.0,30,88,76,82,217.9,188.1,11.1,10.8,7.9,3630,8.5,3590,8.3,3800,8
8,80,85.0,373,100,60,15,1148,100,169.2,47.0,1.0,154.0,130.0,1.2,0,127.0,-68,23,356.0,574.0,174,28.0,34,77,89,95,-118,-238.0,30,91,78,73,218.4,187.6,11.1,10.9,8.8,3660,8.6,3780,8.7,3800,8
9,83,89.0,355,100,60,15,1148,100,169.2,47.0,1.0,144.0,139.0,1.5,0,124.0,-76,21,346.0,562.0,168,29.0,37,76,93,99,-121,-237.0,30,91,77,77,219.4,188.7,11.1,10.9,8.7,3660,8.6,3780,8.7,3800,8


# CELL 4) 모델/스케일러 로드 (FROZEN)

In [5]:
# %%
# ---- scalers
scaler_x_res = joblib.load(SCALER_X_RES_PATH)  # fit on [u, mon, residue_prev]
scaler_y_res = joblib.load(SCALER_Y_RES_PATH)  # fit on residue

scaler_x_bla = joblib.load(SCALER_X_BLA_PATH)  # fit on [u, mon, blaine_prev]
scaler_y_bla = joblib.load(SCALER_Y_BLA_PATH)  # fit on blaine

# ---- quality models
quality_res = tf.keras.models.load_model(QUALITY_RES_PATH)
quality_bla = tf.keras.models.load_model(QUALITY_BLA_PATH)
quality_res.trainable = False
quality_bla.trainable = False

# ---- policy nets
policy_res = tf.keras.models.load_model(POLICY_RES_PATH)
policy_bla = tf.keras.models.load_model(POLICY_BLA_PATH)
policy_res.trainable = False
policy_bla.trainable = False

print("Loaded OK:")
print("- quality_res:", QUALITY_RES_PATH, "in/out:", quality_res.input_shape, quality_res.output_shape)
print("- quality_bla:", QUALITY_BLA_PATH, "in/out:", quality_bla.input_shape, quality_bla.output_shape)
print("- policy_res :", POLICY_RES_PATH,  "out:", policy_res.output_shape)
print("- policy_bla :", POLICY_BLA_PATH,  "out:", policy_bla.output_shape)
print("- scalers: OK")


Loaded OK:
- quality_res: best_model_residue.h5 in/out: (None, 38) (None, 1)
- quality_bla: best_model_blaine.h5 in/out: (None, 38) (None, 1)
- policy_res : policy_residue.keras out: (None, 9)
- policy_bla : policy_blaine.keras out: (None, 9)
- scalers: OK


# CELL5) BUILD TENSORS: RESIDUE용 / BLAINE용 Xp_full(40) 만들기

In [6]:
# %%
def build_Xp_full_for_one_quality(
    df_raw: pd.DataFrame,
    scaler_x, scaler_y,
    prev_col: str, cur_col: str, tgt_col: str
):
    """
    Xp_full layout (N,40):
      [mon_norm(28), qcur_norm(1), u_cur_norm(9), yT_norm(1), qprev_norm(1)]

    - u_cur_norm, mon_norm, qprev_norm은 scaler_x가 fit된 38차원 공간([u,mon,qprev])을 통해 생성
    - qcur_norm, yT_norm은 scaler_y로 생성
    """
    u_cur_raw = df_raw[CONTROL_COLS_9].to_numpy(np.float32)      # (N,9)
    mon_raw   = df_raw[MONITOR_COLS_28].to_numpy(np.float32)     # (N,28)
    qprev_raw = df_raw[[prev_col]].to_numpy(np.float32)          # (N,1)
    qcur_raw  = df_raw[[cur_col]].to_numpy(np.float32)           # (N,1)
    yT_raw    = df_raw[[tgt_col]].to_numpy(np.float32)           # (N,1)

    Xq_raw = np.concatenate([u_cur_raw, mon_raw, qprev_raw], axis=1).astype(np.float32)  # (N,38)
    Xq_norm = scaler_x.transform(Xq_raw).astype(np.float32)                               # (N,38)

    u_cur_norm = Xq_norm[:, :9]
    mon_norm   = Xq_norm[:, 9:9+28]
    qprev_norm = Xq_norm[:, 9+28:9+28+1]

    qcur_norm = scaler_y.transform(qcur_raw).astype(np.float32)
    yT_norm   = scaler_y.transform(yT_raw).astype(np.float32)

    Xp_full = np.concatenate([mon_norm, qcur_norm, u_cur_norm, yT_norm, qprev_norm], axis=1).astype(np.float32)  # (N,40)
    Xp_no_prev = Xp_full[:, :39].astype(np.float32)  # (N,39)

    return Xp_full, Xp_no_prev, yT_norm

# residue tensors
Xp_full_res, Xp39_res, yT_res_norm = build_Xp_full_for_one_quality(
    df, scaler_x_res, scaler_y_res,
    prev_col=RES_PREV_COL, cur_col=RES_CUR_COL, tgt_col=RES_TGT_COL
)

# blaine tensors
Xp_full_bla, Xp39_bla, yT_bla_norm = build_Xp_full_for_one_quality(
    df, scaler_x_bla, scaler_y_bla,
    prev_col=BLA_PREV_COL, cur_col=BLA_CUR_COL, tgt_col=BLA_TGT_COL
)

print("Xp_full_res:", Xp_full_res.shape, "yT_res_norm:", yT_res_norm.shape)
print("Xp_full_bla:", Xp_full_bla.shape, "yT_bla_norm:", yT_bla_norm.shape)


Xp_full_res: (60572, 40) yT_res_norm: (60572, 1)
Xp_full_bla: (60572, 40) yT_bla_norm: (60572, 1)


# CELL 6) DATASET 생성 (res/bla 동일 인덱스 유지가 핵심)

In [7]:
# %%
# 같은 df에서 만들었으니 row alignment는 맞습니다.
idx = np.arange(len(df))
# 평가용이면 split 안 하고 그대로 써도 됨. (원하면 train/val/test split 적용 가능)

ds_res = tf.data.Dataset.from_tensor_slices((Xp_full_res, yT_res_norm)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
ds_bla = tf.data.Dataset.from_tensor_slices((Xp_full_bla, yT_bla_norm)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Datasets ready:", ds_res, ds_bla)


Datasets ready: <PrefetchDataset element_spec=(TensorSpec(shape=(None, 40), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))> <PrefetchDataset element_spec=(TensorSpec(shape=(None, 40), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None))>


# CELL 7) 혼합정책 1-STEP 평가 함수

In [8]:
# %%
def stats_dict(x: np.ndarray, percentiles=(50, 95)):
    x = np.asarray(x).reshape(-1)
    out = {
        "mean": float(np.mean(x)),
        "min":  float(np.min(x)),
        "max":  float(np.max(x)),
        "std":  float(np.std(x)),
    }
    for p in percentiles:
        out[f"p{p}"] = float(np.percentile(x, p))
    return out


def eval_mixed_policy_1step(
    dataset_res, dataset_bla,
    policy_res, policy_bla,
    quality_res, quality_bla,
    scaler_y_res, scaler_y_bla,
    DELTA_MAX_TF,
    alpha=0.5,                       # float or callable(res_before, bla_before)->alpha
    n_skip_batches=0,
    n_batches=5,
    percentiles=(50, 95),
    # specs
    res_min=7.0, res_max=9.0,
    bla_min=3700.0, bla_max=3900.0
):
    d_res_all, d_bla_all = [], []
    imp_res_all, imp_bla_all = [], []
    spec_before_all, spec_after_all = [], []

    it_res = dataset_res.skip(n_skip_batches).take(n_batches)
    it_bla = dataset_bla.skip(n_skip_batches).take(n_batches)

    for (Xp_res, yT_res_norm), (Xp_bla, yT_bla_norm) in zip(it_res, it_bla):
        # --- parse (B,40)
        mon_res   = Xp_res[:, 0:28]
        u_cur_res = Xp_res[:, 29:38]
        qprev_res = Xp_res[:, 39:40]
        Xp39_res  = Xp_res[:, 0:39]

        mon_bla   = Xp_bla[:, 0:28]
        u_cur_bla = Xp_bla[:, 29:38]
        qprev_bla = Xp_bla[:, 39:40]
        Xp39_bla  = Xp_bla[:, 0:39]

        # --- BEFORE
        y_res_before_norm = quality_res(tf.concat([u_cur_res, mon_res, qprev_res], axis=1), training=False)
        y_bla_before_norm = quality_bla(tf.concat([u_cur_bla, mon_bla, qprev_bla], axis=1), training=False)

        # --- policy deltas (norm)
        d_raw_res = policy_res(Xp39_res, training=False)   # (B,9) in [-1,1]
        d_raw_bla = policy_bla(Xp39_bla, training=False)

        d_res_norm = d_raw_res * DELTA_MAX_TF
        d_bla_norm = d_raw_bla * DELTA_MAX_TF

        # --- alpha (fixed or per-sample)
        if callable(alpha):
            y_res_before = scaler_y_res.inverse_transform(y_res_before_norm.numpy()).reshape(-1)
            y_bla_before = scaler_y_bla.inverse_transform(y_bla_before_norm.numpy()).reshape(-1)
            a = np.array([float(alpha(r, b)) for r, b in zip(y_res_before, y_bla_before)], dtype=np.float32).reshape(-1, 1)
            a = tf.constant(a, dtype=tf.float32)  # (B,1)
        else:
            a = tf.constant(float(alpha), dtype=tf.float32)

        # broadcast to (B,9)
        if len(a.shape) == 2:
            d_mix = a * d_bla_norm + (1.0 - a) * d_res_norm
        else:
            d_mix = a * d_bla_norm + (1.0 - a) * d_res_norm

        # --- apply (use residue u_cur as base; MUST align with blaine u_cur in data)
        u_new = tf.clip_by_value(u_cur_res + d_mix, 0.0, 1.0)

        # --- AFTER
        y_res_after_norm = quality_res(tf.concat([u_new, mon_res, qprev_res], axis=1), training=False)
        y_bla_after_norm = quality_bla(tf.concat([u_new, mon_bla, qprev_bla], axis=1), training=False)

        # --- real scale
        y_res_before = scaler_y_res.inverse_transform(y_res_before_norm.numpy()).reshape(-1)
        y_res_after  = scaler_y_res.inverse_transform(y_res_after_norm.numpy()).reshape(-1)

        y_bla_before = scaler_y_bla.inverse_transform(y_bla_before_norm.numpy()).reshape(-1)
        y_bla_after  = scaler_y_bla.inverse_transform(y_bla_after_norm.numpy()).reshape(-1)

        yT_res = scaler_y_res.inverse_transform(yT_res_norm.numpy()).reshape(-1)
        yT_bla = scaler_y_bla.inverse_transform(yT_bla_norm.numpy()).reshape(-1)

        # --- 변화량 (원하시는 “평균적으로 어떻게 변하나”)
        d_res_all.append(y_res_after - y_res_before)
        d_bla_all.append(y_bla_after - y_bla_before)

        # --- target 절대오차 개선량 (양수면 개선)
        imp_res_all.append(np.abs(y_res_before - yT_res) - np.abs(y_res_after - yT_res))
        imp_bla_all.append(np.abs(y_bla_before - yT_bla) - np.abs(y_bla_after - yT_bla))

        # --- “둘 다” spec 만족 여부
        spec_before = (res_min <= y_res_before) & (y_res_before <= res_max) & (bla_min <= y_bla_before) & (y_bla_before <= bla_max)
        spec_after  = (res_min <= y_res_after)  & (y_res_after  <= res_max) & (bla_min <= y_bla_after)  & (y_bla_after  <= bla_max)
        spec_before_all.append(spec_before.astype(np.float32))
        spec_after_all.append(spec_after.astype(np.float32))

    d_res_all = np.concatenate(d_res_all)
    d_bla_all = np.concatenate(d_bla_all)
    imp_res_all = np.concatenate(imp_res_all)
    imp_bla_all = np.concatenate(imp_bla_all)
    spec_before_all = np.concatenate(spec_before_all)
    spec_after_all  = np.concatenate(spec_after_all)

    report = {
        "delta_res(after-before)": stats_dict(d_res_all, percentiles),
        "delta_bla(after-before)": stats_dict(d_bla_all, percentiles),
        "improve_abs_err_res":     stats_dict(imp_res_all, percentiles),
        "improve_abs_err_bla":     stats_dict(imp_bla_all, percentiles),
        "spec_both_before_rate(%)": float(np.mean(spec_before_all) * 100.0),
        "spec_both_after_rate(%)":  float(np.mean(spec_after_all) * 100.0),
    }
    return report


# CELL 8) DELTA_MAX_TF + alpha 스윕 실행

In [9]:
# %%
DELTA_MAX = np.array([DELTA_MAX_DEFAULT] * 9, dtype=np.float32).reshape(1, -1)
DELTA_MAX_TF = tf.constant(DELTA_MAX, dtype=tf.float32)

for a in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    rep = eval_mixed_policy_1step(
        dataset_res=ds_res,
        dataset_bla=ds_bla,
        policy_res=policy_res,
        policy_bla=policy_bla,
        quality_res=quality_res,
        quality_bla=quality_bla,
        scaler_y_res=scaler_y_res,
        scaler_y_bla=scaler_y_bla,
        DELTA_MAX_TF=DELTA_MAX_TF,
        alpha=a,
        n_skip_batches=N_SKIP_BATCHES,
        n_batches=N_BATCHES,
        percentiles=(50, 95),
        res_min=RES_MIN, res_max=RES_MAX,
        bla_min=BLA_MIN, bla_max=BLA_MAX
    )
    print("\n==============================")
    print("alpha =", a)
    print("delta_res:", rep["delta_res(after-before)"])
    print("delta_bla:", rep["delta_bla(after-before)"])
    print("improve_err_res:", rep["improve_abs_err_res"])
    print("improve_err_bla:", rep["improve_abs_err_bla"])
    print("spec both before/after (%):", rep["spec_both_before_rate(%)"], "->", rep["spec_both_after_rate(%)"])



alpha = 0.0
delta_res: {'mean': -0.4900686740875244, 'min': -2.346987724304199, 'max': 1.2758121490478516, 'std': 0.42675915360450745, 'p50': -0.4643566608428955, 'p95': 0.14888360500335665}
delta_bla: {'mean': -0.46641770005226135, 'min': -490.964111328125, 'max': 331.322998046875, 'std': 65.53007507324219, 'p50': 2.1246337890625, 'p95': 104.61081542968743}
improve_err_res: {'mean': 0.5035795569419861, 'min': -0.4812326431274414, 'max': 2.346987724304199, 'std': 0.3741752505302429, 'p50': 0.4433257579803467, 'p95': 1.2106258630752562}
improve_err_bla: {'mean': 8.73376178741455, 'min': -252.015625, 'max': 434.159423828125, 'std': 59.41219711303711, 'p50': 8.1046142578125, 'p95': 106.18730468749997}
spec both before/after (%): 46.44531309604645 -> 53.242188692092896

alpha = 0.2
delta_res: {'mean': -0.4677475094795227, 'min': -2.3100948333740234, 'max': 1.22965669631958, 'std': 0.4282884895801544, 'p50': -0.4386005401611328, 'p95': 0.17285959720611568}
delta_bla: {'mean': 21.8912982940

# CELL 9) RESIDUE 스펙 우선 가변 alpha 규칙으로 실행

In [14]:
# %%
def alpha_rule_res_first(res_before, bla_before,
                         res_min=7.0, res_max=9.0,
                         alpha_when_res_out=0.1,
                         alpha_when_res_in=0.7):
    res_in = (res_min <= res_before <= res_max)
    return alpha_when_res_in if res_in else alpha_when_res_out


rep = eval_mixed_policy_1step(
    dataset_res=ds_res,
    dataset_bla=ds_bla,
    policy_res=policy_res,
    policy_bla=policy_bla,
    quality_res=quality_res,
    quality_bla=quality_bla,
    scaler_y_res=scaler_y_res,
    scaler_y_bla=scaler_y_bla,
    DELTA_MAX_TF=DELTA_MAX_TF,
    alpha=alpha_rule_res_first,  # callable
    n_skip_batches=N_SKIP_BATCHES,
    n_batches=N_BATCHES,
    percentiles=(50, 95),
    res_min=RES_MIN, res_max=RES_MAX,
    bla_min=BLA_MIN, bla_max=BLA_MAX
)

print(rep)


{'delta_res(after-before)': {'mean': -0.451442152261734, 'min': -2.377474308013916, 'max': 1.2637758255004883, 'std': 0.43773534893989563, 'p50': -0.3996772766113281, 'p95': 0.18996496200561477}, 'delta_bla(after-before)': {'mean': 63.232757568359375, 'min': -482.479736328125, 'max': 375.51025390625, 'std': 67.9433822631836, 'p50': 67.8753662109375, 'p95': 165.68215332031247}, 'improve_abs_err_res': {'mean': 0.35312482714653015, 'min': -1.327667236328125, 'max': 2.3326034545898438, 'std': 0.4241907596588135, 'p50': 0.2818589210510254, 'p95': 1.2018603563308716}, 'improve_abs_err_bla': {'mean': 70.29830932617188, 'min': -125.831298828125, 'max': 442.643798828125, 'std': 57.25984573364258, 'p50': 68.5419921875, 'p95': 165.131591796875}, 'spec_both_before_rate(%)': 46.44531309604645, 'spec_both_after_rate(%)': 86.572265625}


In [15]:
import pandas as pd

def report_to_table_from_rep(rep: dict) -> pd.DataFrame:
    rows = []

    rows.append({
        "Quality": "residue",
        "Δ(after-before) mean": rep["delta_res(after-before)"]["mean"],
        "Δ(after-before) p50":  rep["delta_res(after-before)"]["p50"],
        "Δ(after-before) p95":  rep["delta_res(after-before)"]["p95"],
        "Improvement |err| mean": rep["improve_abs_err_res"]["mean"],
        "Improvement |err| p50":  rep["improve_abs_err_res"]["p50"],
        "Improvement |err| p95":  rep["improve_abs_err_res"]["p95"],
    })

    rows.append({
        "Quality": "blaine",
        "Δ(after-before) mean": rep["delta_bla(after-before)"]["mean"],
        "Δ(after-before) p50":  rep["delta_bla(after-before)"]["p50"],
        "Δ(after-before) p95":  rep["delta_bla(after-before)"]["p95"],
        "Improvement |err| mean": rep["improve_abs_err_bla"]["mean"],
        "Improvement |err| p50":  rep["improve_abs_err_bla"]["p50"],
        "Improvement |err| p95":  rep["improve_abs_err_bla"]["p95"],
    })

    df = pd.DataFrame(rows).set_index("Quality")
    return df


df_report = report_to_table_from_rep(rep)

pd.set_option("display.float_format", "{:+.3f}".format)
display(df_report)


,Δ(after-before) mean,Δ(after-before) p50,Δ(after-before) p95,Improvement |err| mean,Improvement |err| p50,Improvement |err| p95
Quality,,,,,,
residue,-0.451,-0.400,+0.190,+0.353,+0.282,+1.202
blaine,+63.233,+67.875,+165.682,+70.298,+68.542,+165.132
